# Data Integration

## Business Objective

The objective of this notebook is to integrate the cleaned e-commerce datasets into a reliable analytical data model suitable for downstream business analysis, SQL querying, dashboard development, and executive reporting.

Rather than analyzing each table independently, this phase connects the cleaned datasets through their relational keys to create a consistent view of customers, orders, products, reviews, and user behavior.

This notebook focuses on:

- Loading cleaned datasets
- Understanding table relationships
- Validating primary and foreign keys
- Building an order-level analytical dataset
- Building an item-level sales dataset
- Creating supporting behavioral and review datasets
- Exporting integrated datasets for EDA, SQL, and Power BI

## Integration Methodology

The integration process follows a structured workflow:

1. Load cleaned datasets  
2. Inspect dataset dimensions  
3. Define relational schema  
4. Validate primary keys  
5. Validate foreign key relationships  
6. Merge transactional tables  
7. Validate row counts after joins  
8. Create analysis-ready datasets  
9. Export integrated outputs  

The goal is to ensure that all downstream analysis is based on a trustworthy and well-documented analytical data model.

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

## Set Project Directory

In [ ]:
os.chdir("/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis")

PROJECT_ROOT = Path(os.getcwd())
CLEAN_DATA_DIR = PROJECT_ROOT / "data" / "cleaned"
INTEGRATED_DATA_DIR = PROJECT_ROOT / "data" / "integrated"

INTEGRATED_DATA_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

## Load Cleaned Datasets

In [ ]:
users = pd.read_csv(CLEAN_DATA_DIR / "users_clean.csv")
products = pd.read_csv(CLEAN_DATA_DIR / "products_clean.csv")
orders = pd.read_csv(CLEAN_DATA_DIR / "orders_clean.csv")
order_items = pd.read_csv(CLEAN_DATA_DIR / "order_items_clean.csv")
reviews = pd.read_csv(CLEAN_DATA_DIR / "reviews_clean.csv")
events = pd.read_csv(CLEAN_DATA_DIR / "events_clean.csv")

## Convert Date Columns

In [ ]:
users["signup_date"] = pd.to_datetime(users["signup_date"], errors="coerce")
orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
reviews["review_date"] = pd.to_datetime(reviews["review_date"], errors="coerce")
events["event_timestamp"] = pd.to_datetime(events["event_timestamp"], errors="coerce")

# Dataset Overview

Before integrating the datasets, the dimensions of each cleaned table are reviewed to confirm that the cleaned files were loaded correctly.

In [ ]:
datasets = {
    "users": users,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "reviews": reviews,
    "events": events
}

overview = pd.DataFrame({
    "table": list(datasets.keys()),
    "rows": [df.shape[0] for df in datasets.values()],
    "columns": [df.shape[1] for df in datasets.values()]
})

overview

## Interpretation

The cleaned datasets have been successfully loaded and are ready for relational validation. At this stage, no joins have been performed yet. The next step is to define how the tables should connect through their primary and foreign keys.

# Relational Schema

The e-commerce database follows a relational structure.

## Core Relationships

- `users.user_id` connects to `orders.user_id`
- `orders.order_id` connects to `order_items.order_id`
- `products.product_id` connects to `order_items.product_id`
- `users.user_id` connects to `reviews.user_id`
- `products.product_id` connects to `reviews.product_id`
- `users.user_id` connects to `events.user_id`
- `products.product_id` connects to `events.product_id`

## Analytical Model

For business analysis, two primary datasets will be created:

1. **Item-level sales dataset**  
   Combines orders, order items, products, and users.  
   This dataset supports revenue, product, category, customer, and geographic analysis.

2. **Behavioral events dataset**  
   Combines events, users, and products.  
   This dataset supports funnel analysis and customer behavior analysis.

# Primary Key Validation

Before joining tables, each primary key is validated to ensure uniqueness. Duplicate primary keys would create unreliable joins and may inflate row counts during integration.

In [ ]:
primary_key_checks = pd.DataFrame({
    "table": ["users", "products", "orders", "order_items", "reviews", "events"],
    "primary_key": ["user_id", "product_id", "order_id", "order_item_id", "review_id", "event_id"],
    "rows": [
        len(users),
        len(products),
        len(orders),
        len(order_items),
        len(reviews),
        len(events)
    ],
    "unique_primary_keys": [
        users["user_id"].nunique(),
        products["product_id"].nunique(),
        orders["order_id"].nunique(),
        order_items["order_item_id"].nunique(),
        reviews["review_id"].nunique(),
        events["event_id"].nunique()
    ],
    "duplicate_primary_keys": [
        users["user_id"].duplicated().sum(),
        products["product_id"].duplicated().sum(),
        orders["order_id"].duplicated().sum(),
        order_items["order_item_id"].duplicated().sum(),
        reviews["review_id"].duplicated().sum(),
        events["event_id"].duplicated().sum()
    ]
})

primary_key_checks

## Primary Key Interpretation

The primary key validation confirms whether each table can be safely used in relational joins. A duplicate primary key could cause duplicated rows after merging and distort revenue, customer, product, or event metrics.

If all duplicate primary key counts equal zero, the tables are structurally ready for integration.

# Foreign Key Validation

Foreign key validation checks whether records in one table correctly reference records in another table.

This step is important because invalid foreign keys can cause missing values after joins and may indicate broken relationships between tables.

In [ ]:
foreign_key_checks = pd.DataFrame({
    "relationship": [
        "orders.user_id → users.user_id",
        "order_items.order_id → orders.order_id",
        "order_items.product_id → products.product_id",
        "reviews.user_id → users.user_id",
        "reviews.product_id → products.product_id",
        "events.user_id → users.user_id",
        "events.product_id → products.product_id"
    ],
    "foreign_key_rows": [
        orders["user_id"].notna().sum(),
        order_items["order_id"].notna().sum(),
        order_items["product_id"].notna().sum(),
        reviews["user_id"].notna().sum(),
        reviews["product_id"].notna().sum(),
        events["user_id"].notna().sum(),
        events["product_id"].notna().sum()
    ],
    "unmatched_foreign_keys": [
        (~orders["user_id"].isin(users["user_id"])).sum(),
        (~order_items["order_id"].isin(orders["order_id"])).sum(),
        (~order_items["product_id"].isin(products["product_id"])).sum(),
        (~reviews["user_id"].isin(users["user_id"])).sum(),
        (~reviews["product_id"].isin(products["product_id"])).sum(),
        (~events["user_id"].isin(users["user_id"])).sum(),
        (~events["product_id"].isin(products["product_id"])).sum()
    ]
})

foreign_key_checks

## Foreign Key Interpretation

Foreign key validation helps determine whether relationships between tables are reliable. If unmatched foreign keys exist, they should be documented and considered when interpreting joined datasets.

For portfolio purposes, documenting unmatched relationships is important because it demonstrates that joins were not performed blindly.

# Build Item-Level Sales Dataset

The item-level sales dataset combines:

- Order item information
- Order-level transaction details
- Product details
- Customer details

This dataset will support most business analytics use cases, including revenue analysis, product performance, category analysis, customer segmentation, and geographic sales analysis.

## Step 1: Merge Order Items with Orders

In [ ]:
sales_items = order_items.merge(
    orders,
    on="order_id",
    how="left",
    indicator=True
)

sales_items["_merge"].value_counts()

## Validate Order Join

In [ ]:
unmatched_orders = sales_items[sales_items["_merge"] != "both"]

unmatched_orders.shape[0]

## Remove Join Indicator

In [ ]:
sales_items = sales_items.drop(columns="_merge")

## Step 2: Merge Sales Items with Products

In [ ]:
sales_items = sales_items.merge(
    products,
    on="product_id",
    how="left",
    indicator=True
)

sales_items["_merge"].value_counts()

## Validate Product Join

In [ ]:
unmatched_products = sales_items[sales_items["_merge"] != "both"]

unmatched_products.shape[0]

## Remove Join Indicator

In [ ]:
sales_items = sales_items.drop(columns="_merge")

## Step 3: Merge Sales Items with Users

In [ ]:
sales_items = sales_items.merge(
    users,
    on="user_id",
    how="left",
    indicator=True
)

sales_items["_merge"].value_counts()

## Validate User Join

In [ ]:
unmatched_users = sales_items[sales_items["_merge"] != "both"]

unmatched_users.shape[0]

## Remove Join Indicator

In [ ]:
sales_items = sales_items.drop(columns="_merge")

## Create Revenue Column

Although order-level totals are available, item-level revenue is calculated from quantity and item price to support product and category-level analysis.

In [ ]:
sales_items["item_revenue"] = sales_items["quantity"] * sales_items["item_price"]

sales_items[["quantity", "item_price", "item_revenue"]].head()

## Sales Dataset Preview

In [ ]:
sales_items.head()

## Sales Dataset Shape

In [ ]:
sales_items.shape

## Sales Dataset Interpretation

The item-level sales dataset is the primary analytical dataset for revenue and product performance analysis. Each row represents a purchased item within an order, enriched with order, product, and customer attributes.

This dataset will be used heavily in EDA, SQL analysis, and dashboard development.

# Build Reviews Analytical Dataset

The reviews dataset is enriched with product and customer attributes to support customer satisfaction and product feedback analysis.

In [ ]:
reviews_analysis = reviews.merge(
    users,
    on="user_id",
    how="left"
).merge(
    products,
    on="product_id",
    how="left"
)

reviews_analysis.shape

## Reviews Dataset Preview

In [ ]:
reviews_analysis.head()

## Reviews Dataset Interpretation

The integrated reviews dataset connects customer feedback to both customer demographics and product attributes. This enables analysis of customer satisfaction by product category, city, customer segment, and product rating patterns.

# Build Events Analytical Dataset

The events dataset is enriched with customer and product information to support behavioral analysis and conversion funnel analysis.

In [ ]:
events_analysis = events.merge(
    users,
    on="user_id",
    how="left"
).merge(
    products,
    on="product_id",
    how="left"
)

events_analysis.shape

## Events Dataset Preview

In [ ]:
events_analysis.head()

## Events Dataset Interpretation

The integrated events dataset connects user behavior to customer and product attributes. This dataset can support conversion funnel analysis, product engagement analysis, and user journey exploration.

# Data Model Validation

This section validates the integrated datasets by checking row counts, missing values, and duplicate records after joins.

In [ ]:
integrated_validation = pd.DataFrame({
    "dataset": ["sales_items", "reviews_analysis", "events_analysis"],
    "rows": [
        len(sales_items),
        len(reviews_analysis),
        len(events_analysis)
    ],
    "columns": [
        sales_items.shape[1],
        reviews_analysis.shape[1],
        events_analysis.shape[1]
    ],
    "duplicate_rows": [
        sales_items.duplicated().sum(),
        reviews_analysis.duplicated().sum(),
        events_analysis.duplicated().sum()
    ],
    "missing_values_total": [
        sales_items.isna().sum().sum(),
        reviews_analysis.isna().sum().sum(),
        events_analysis.isna().sum().sum()
    ]
})

integrated_validation

## Validation Interpretation

The validation summary confirms that integrated datasets were created successfully. Remaining missing values may be expected due to missing values preserved from the cleaning phase or unmatched relationships identified during foreign key validation.

These integrated datasets are suitable for exploratory analysis because major structural issues such as duplicate primary keys and exact duplicate records were already addressed during cleaning.

# Export Integrated Datasets

The integrated datasets are exported to the `data/integrated` directory for use in EDA, SQL analysis, Power BI, and reporting.

In [ ]:
sales_items.to_csv(INTEGRATED_DATA_DIR / "sales_items_analysis.csv", index=False)
reviews_analysis.to_csv(INTEGRATED_DATA_DIR / "reviews_analysis.csv", index=False)
events_analysis.to_csv(INTEGRATED_DATA_DIR / "events_analysis.csv", index=False)

list(INTEGRATED_DATA_DIR.iterdir())

# Executive Summary

## Objective

The objective of this notebook was to integrate the cleaned e-commerce datasets into analysis-ready datasets suitable for business analysis and visualization.

## Integration Work Completed

The following work was completed:

- Loaded all cleaned datasets from the `data/cleaned` directory.
- Validated primary key uniqueness across all core tables.
- Validated foreign key relationships between related tables.
- Created an item-level sales dataset by joining order items, orders, products, and users.
- Created an enriched reviews dataset by joining reviews with users and products.
- Created an enriched events dataset by joining events with users and products.
- Created an item-level revenue field to support product and category-level sales analysis.
- Validated integrated datasets after joins.
- Exported integrated analytical datasets to the `data/integrated` directory.

## Final Analytical Outputs

The following integrated datasets were created:

- `sales_items_analysis.csv`
- `reviews_analysis.csv`
- `events_analysis.csv`

## Business Value

The integrated data model transforms separate cleaned CSV files into structured analytical datasets. This allows the project to move from data preparation into business insight generation.

The item-level sales dataset supports revenue, product, customer, category, and geographic analysis. The reviews dataset supports customer satisfaction analysis. The events dataset supports customer behavior and funnel analysis.

## Next Steps

The next stage of the project is Exploratory Data Analysis (EDA). In that phase, the integrated datasets will be used to answer business questions such as:

- Which categories generate the most revenue?
- Which products are top performers?
- How does revenue change over time?
- Which cities contain the most customers?
- How do product ratings relate to sales?
- Where do users drop off in the behavioral funnel?

This integration phase ensures that all downstream analysis is based on a documented and validated analytical data model.